# Q-Mini — Training-Ready Model Builder (Build now, train later)

This notebook prepares one complete **0.7M-class, training-ready Q-Mini package** in Google Drive.

It will:
- use the repository's from-scratch CausalConv + gated diagonal SSM + sparse Top-1 MoE model;
- keep Transformer self-attention and Q/K/V attention out of the architecture;
- fit the Hugging Face Tokenizers byte-level BPE tokenizer to the corpus you place in Drive;
- create fingerprinted train/validation token streams and the required loss masks;
- build the full random-initialized model and export Q2-v2 packed matrix weights;
- verify the model, tokenizer, data manifests and Drive copy before publishing the package.

**This notebook does not train the neural model.** It does not create an optimizer, run backward passes or update model weights. Tokenizer fitting and dataset preparation are preprocessing, not neural-model training.

The default 0.7M-class profile is the Colab training target. The repository's 5B profile can be initialized as a packed artifact by the separate 5B builder, but the current trainer does **not** implement FSDP/ZeRO or 5B multi-GPU training. Do not use a 5B artifact with this single-device training path.

In [ ]:
# ---------------- USER SETTINGS ----------------
MODEL_CONFIG_REL = "configs/qmini_light_0.7m.json"
DATA_REL = "QMini-Light-0.7M/datasets"
ART_REL = "QMini-Light-0.7M"

VOCAB_SIZE = 4096
VALIDATION_FRACTION = 0.02
NEAR_DUPLICATE_HAMMING_DISTANCE = 0
LOSS_MASK_MODE = "auto"  # auto = assistant-only for chat/Q&A, full LM for plain text

# Optional strict dataset-release gate. Supply a qmini-source-provenance-v1 JSON
# file in Google Drive before setting RELEASE_MODE=True.
SOURCE_METADATA_REL = None
RELEASE_MODE = False

# The builder is build-only. Training is activated separately from
# QMini_Training_Runner.ipynb after the model package has been verified.
TRAIN_SEQUENCE_LENGTH = 512

print("Model profile:", MODEL_CONFIG_REL)
print("Neural model training enabled: False (separate runner only)")

In [ ]:
# ---------------- GOOGLE DRIVE + SOURCE CHECKOUT ----------------
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import base64
import getpass
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time

MY_DRIVE = Path("/content/drive/MyDrive")
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive is not mounted at /content/drive/MyDrive.")

DATA_DIR = MY_DRIVE / DATA_REL
ART_ROOT = MY_DRIVE / ART_REL
DATA_DIR.mkdir(parents=True, exist_ok=True)
ART_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/Q-Mini-Light-0.7M-Training-Ready")
REPOSITORY_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"

def run_checked(command, *, cwd=None, capture_output=False, env=None):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        check=True,
        text=True,
        capture_output=capture_output,
        env=env,
    )


def private_repo_git_env():
    token = os.environ.get("QMINI_GITHUB_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("QMINI_GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        token = getpass.getpass(
            "GitHub read-only token for private Q-Mini-Light repo (hidden input): "
        )
    if not token:
        raise RuntimeError(
            "A read-only GitHub token for lion36508/Q-mini-light-0.7M is required."
        )
    env = os.environ.copy()
    env.pop("QMINI_GITHUB_TOKEN", None)
    basic = base64.b64encode(f"x-access-token:{token}".encode("utf-8")).decode("ascii")
    env["GIT_CONFIG_COUNT"] = "1"
    env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {basic}"
    return env

GIT_ENV = private_repo_git_env()
try:
    if REPO_DIR.exists():
        run_checked(["git", "fetch", "--depth", "1", "origin", "main"], cwd=REPO_DIR, env=GIT_ENV)
        run_checked(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR, env=GIT_ENV)
    else:
        run_checked(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, REPO_DIR], env=GIT_ENV)
finally:
    GIT_ENV.pop("GIT_CONFIG_VALUE_0", None)
    del GIT_ENV

SOURCE_REVISION = run_checked(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True
).stdout.strip()

MODEL_CONFIG_PATH = REPO_DIR / MODEL_CONFIG_REL
if not MODEL_CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Model config not found: {MODEL_CONFIG_PATH}")

print("Source revision:", SOURCE_REVISION)
print("Corpus folder:", DATA_DIR)
print("Drive artifact root:", ART_ROOT)

In [ ]:
# ---------------- INSTALL OPEN-SOURCE DEPENDENCIES ----------------
# Keep Colab's preinstalled CUDA-enabled PyTorch wheel. Do not replace it.
run_checked([
    sys.executable, "-m", "pip", "install",
    "numpy>=1.24", "tokenizers>=0.15", "pyarrow>=15.0", "packaging>=23",
])
run_checked([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."], cwd=REPO_DIR)

import torch
from packaging.version import Version

torch_version = Version(torch.__version__.split("+", 1)[0])
if torch_version < Version("2.10.0"):
    raise RuntimeError(
        f"Q-Mini requires PyTorch >=2.10.0; Colab has {torch.__version__}. "
        "Select a compatible runtime instead of blindly replacing the CUDA wheel."
    )

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Build-only mode is CPU-compatible; later neural-model training requires a suitable GPU.")

In [ ]:
# ---------------- DATASET FINGERPRINT + PACKAGE ID ----------------
from qmini.config import QMiniConfig

base_config = QMiniConfig.from_json(MODEL_CONFIG_PATH)
if base_config.max_context != 512:
    raise RuntimeError("The Light profile must use the requested 512-token context.")
if TRAIN_SEQUENCE_LENGTH != base_config.max_context:
    raise RuntimeError("TRAIN_SEQUENCE_LENGTH must match the Light model context.")
if base_config.estimated_parameter_count() > 1_000_000:
    raise RuntimeError(
        "This notebook is deliberately limited to the <=1M Light profile. "
        "Only the 0.7M Light profile is selected here; 5B training remains in the separate Q-Mini repository."
    )

if RELEASE_MODE and not SOURCE_METADATA_REL:
    raise ValueError("RELEASE_MODE=True requires SOURCE_METADATA_REL to point to a source/license manifest.")

SOURCE_METADATA_PATH = (MY_DRIVE / SOURCE_METADATA_REL) if SOURCE_METADATA_REL else None
if SOURCE_METADATA_PATH is not None and not SOURCE_METADATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset source metadata not found: {SOURCE_METADATA_PATH}")

data_files = sorted(
    path for path in DATA_DIR.rglob("*")
    if path.is_file() and not any(part.startswith(".") for part in path.relative_to(DATA_DIR).parts)
)
if not data_files:
    raise FileNotFoundError(
        f"No corpus files found in {DATA_DIR}. Put your reviewed text/JSONL/Parquet corpus there, then rerun."
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_records = []
for path in data_files:
    source_records.append({
        "relative_path": path.relative_to(DATA_DIR).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

source_metadata_sha256 = (
    sha256_file(SOURCE_METADATA_PATH) if SOURCE_METADATA_PATH is not None else None
)
package_identity = {
    "format": "qmini-training-ready-package-id-v1",
    "source_revision": SOURCE_REVISION,
    "model_config": base_config.to_dict(),
    "dataset_files": source_records,
    "source_metadata_sha256": source_metadata_sha256,
    "tokenizer": {
        "type": "byte-level-bpe",
        "vocab_size_target": VOCAB_SIZE,
        "validation_fraction": VALIDATION_FRACTION,
        "near_duplicate_hamming_distance": NEAR_DUPLICATE_HAMMING_DISTANCE,
    },
    "preparation": {
        "loss_mask_mode": LOSS_MASK_MODE,
        "minimum_sequence_length": TRAIN_SEQUENCE_LENGTH,
    },
    "release_mode": RELEASE_MODE,
}
PACKAGE_KEY = hashlib.sha256(
    json.dumps(package_identity, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()[:16]

PACKAGE_NAME = f"QMini-Light-0.7M-Q2v2-{PACKAGE_KEY}"
LOCAL_PACKAGE_DIR = Path("/content") / PACKAGE_NAME
FINAL_PACKAGE_DIR = ART_ROOT / "training_ready" / PACKAGE_NAME
FINAL_PACKAGE_DIR.parent.mkdir(parents=True, exist_ok=True)

print("Corpus files:", len(source_records))
print("Corpus bytes:", f"{sum(item['size_bytes'] for item in source_records):,}")
print("Package identity:", PACKAGE_KEY)
print("Final Drive package:", FINAL_PACKAGE_DIR)

In [ ]:
# ---------------- TOKENIZER + TOKEN STREAMS + FULL MODEL BUILD ----------------
from qmini.tokenizer import load_tokenizer
from qmini.text import SPECIAL_TOKENS
from qmini.export import verify_packed_export
from qmini.train import (
    _verify_initial_brain_artifact,
    _verify_prepared_dataset_manifest,
    _load_tokens,
    _validate_token_ids,
)

def verify_package(package_dir):
    package_dir = Path(package_dir)
    tokenizer_path = package_dir / "tokenizer" / "tokenizer.json"
    tokenizer_sidecar = Path(str(tokenizer_path) + ".qmini.json")
    dataset_dir = package_dir / "dataset"
    initial_dir = package_dir / "initial_brain"
    config_path = initial_dir / "config.json"
    model_path = initial_dir / "model.qmini.npz"
    manifest_path = initial_dir / "model_manifest.json"
    runtime_dir = package_dir / "training_runtime"
    runtime_manifest_path = runtime_dir / "TRAINING_RUNTIME.json"
    progress_path = package_dir / "training_state" / "training_progress.json"

    active_config_path = package_dir / "active_config.json"
    required = [
        tokenizer_path, tokenizer_sidecar, active_config_path, dataset_dir / "train.bin",
        dataset_dir / "valid.bin", dataset_dir / "train.mask.bin",
        dataset_dir / "valid.mask.bin", dataset_dir / "dataset.json",
        config_path, model_path, manifest_path,
        runtime_manifest_path, runtime_dir / "qmini" / "train.py",
        runtime_dir / "qmini" / "checkpointing.py", progress_path,
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("Training-ready package is incomplete: " + ", ".join(missing))

    config = QMiniConfig.from_json(config_path)
    if QMiniConfig.from_json(active_config_path).to_dict() != config.to_dict():
        raise RuntimeError("Root active_config.json differs from initial_brain/config.json.")
    tokenizer = load_tokenizer(tokenizer_path)
    if tokenizer.get_vocab_size() != config.vocab_size:
        raise RuntimeError("Tokenizer vocabulary size does not match the model config.")
    for token in SPECIAL_TOKENS:
        if tokenizer.token_to_id(token) is None:
            raise RuntimeError(f"Tokenizer is missing required structural token: {token}")

    packed_report = verify_packed_export(model_path)
    if (
        packed_report.get("format") != "qmini-packed-v2"
        or packed_report.get("q2_version") != 2
        or packed_report.get("parameter_count") != config.estimated_parameter_count()
    ):
        raise RuntimeError("Q2-v2 archive failed format/version/parameter verification.")

    _verify_initial_brain_artifact(model_path, config)
    _verify_prepared_dataset_manifest(
        dataset_dir / "train.bin", dataset_dir / "valid.bin",
        tokenizer_path, dataset_dir / "train.mask.bin",
        dataset_dir / "valid.mask.bin", config.vocab_size,
    )
    train_tokens = _load_tokens(dataset_dir / "train.bin", "Training")
    valid_tokens = _load_tokens(dataset_dir / "valid.bin", "Validation")
    _validate_token_ids(train_tokens, config.vocab_size, "Training")
    _validate_token_ids(valid_tokens, config.vocab_size, "Validation")
    if len(train_tokens) < TRAIN_SEQUENCE_LENGTH + 2:
        raise ValueError("Training corpus is too short for TRAIN_SEQUENCE_LENGTH.")
    if len(valid_tokens) < TRAIN_SEQUENCE_LENGTH + 2:
        raise ValueError("Validation corpus is too short for TRAIN_SEQUENCE_LENGTH.")

    runtime_manifest = json.loads(runtime_manifest_path.read_text(encoding="utf-8"))
    if runtime_manifest.get("enabled") is not False or runtime_manifest.get("requires_manual_activation") is not True:
        raise RuntimeError("Training runtime must be packaged but disabled by default.")
    for record in runtime_manifest.get("source_files", []):
        relative = Path(record["path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Training runtime manifest contains an unsafe source path.")
        source_file = runtime_dir / relative
        if not source_file.is_file() or sha256_file(source_file) != record.get("sha256"):
            raise RuntimeError(f"Training runtime source hash mismatch: {relative}")
    progress = json.loads(progress_path.read_text(encoding="utf-8"))
    if progress.get("status") != "not_started" or progress.get("current_step") != 0:
        raise RuntimeError("A fresh package must start with training disabled at step zero.")

    model_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if model_manifest.get("training_runtime_manifest_sha256") != sha256_file(runtime_manifest_path):
        raise RuntimeError("Training runtime manifest does not match the root model-manifest fingerprint.")
    if model_manifest.get("training_steps") != 0 or model_manifest.get("optimizer_steps") != 0:
        raise RuntimeError("Initial brain unexpectedly contains model-training updates.")
    if model_manifest.get("matrix_weight_bits") != 2:
        raise RuntimeError("Initial brain is not a Q2-v2 two-bit matrix-weight artifact.")

    return {
        "config": config,
        "packed_report": packed_report,
        "model_manifest": model_manifest,
        "tokenizer_sha256": sha256_file(tokenizer_path),
        "dataset_manifest": json.loads((dataset_dir / "dataset.json").read_text(encoding="utf-8")),
        "train_tokens": len(train_tokens),
        "valid_tokens": len(valid_tokens),
    }

if FINAL_PACKAGE_DIR.is_dir():
    print("A package with this exact source/config fingerprint already exists. Verifying; no overwrite will occur.")
    PACKAGE_DIR = FINAL_PACKAGE_DIR
    verified = verify_package(PACKAGE_DIR)
else:
    if LOCAL_PACKAGE_DIR.exists():
        raise FileExistsError(
            f"Local staging folder already exists: {LOCAL_PACKAGE_DIR}. "
            "Reconnect to a fresh Colab runtime rather than deleting an unverified build."
        )

    LOCAL_PACKAGE_DIR.mkdir(parents=True, exist_ok=False)
    TOKENIZER_DIR = LOCAL_PACKAGE_DIR / "tokenizer"
    DATASET_DIR = LOCAL_PACKAGE_DIR / "dataset"
    INITIAL_DIR = LOCAL_PACKAGE_DIR / "initial_brain"
    TOKENIZER_DIR.mkdir()

    TOKENIZER_PATH = TOKENIZER_DIR / "tokenizer.json"
    TOKENIZER_CONFIG_PATH = LOCAL_PACKAGE_DIR / "active_config.json"

    tokenizer_command = [
        sys.executable, "-m", "qmini.cli", "tokenizer",
        "--data-dir", str(DATA_DIR),
        "--output", str(TOKENIZER_PATH),
        "--vocab-size", str(VOCAB_SIZE),
        "--base-config", str(MODEL_CONFIG_PATH),
        "--valid-fraction", str(VALIDATION_FRACTION),
        "--near-duplicate-hamming-distance", str(NEAR_DUPLICATE_HAMMING_DISTANCE),
        "--config-output", str(TOKENIZER_CONFIG_PATH),
    ]
    if SOURCE_METADATA_PATH is not None:
        tokenizer_command.extend(["--source-metadata", str(SOURCE_METADATA_PATH)])
    if RELEASE_MODE:
        tokenizer_command.append("--release-mode")
    run_checked(tokenizer_command, cwd=REPO_DIR)

    active_config = QMiniConfig.from_json(TOKENIZER_CONFIG_PATH)
    active_values = active_config.to_dict()
    base_values = base_config.to_dict()
    base_values["vocab_size"] = active_config.vocab_size
    if active_values != base_values:
        raise RuntimeError("Tokenizer creation unexpectedly changed architecture fields other than vocab_size.")

    tokenizer = load_tokenizer(TOKENIZER_PATH)
    if tokenizer.get_vocab_size() != active_config.vocab_size:
        raise RuntimeError("Tokenizer and active model config vocabulary do not match.")
    for token in SPECIAL_TOKENS:
        if tokenizer.token_to_id(token) is None:
            raise RuntimeError(f"Tokenizer is missing required structural token: {token}")

    prepare_command = [
        sys.executable, "-m", "qmini.cli", "prepare",
        "--data-dir", str(DATA_DIR),
        "--tokenizer", str(TOKENIZER_PATH),
        "--output-dir", str(DATASET_DIR),
        "--valid-fraction", str(VALIDATION_FRACTION),
        "--near-duplicate-hamming-distance", str(NEAR_DUPLICATE_HAMMING_DISTANCE),
        "--loss-mask", LOSS_MASK_MODE,
    ]
    if SOURCE_METADATA_PATH is not None:
        prepare_command.extend(["--source-metadata", str(SOURCE_METADATA_PATH)])
    if RELEASE_MODE:
        prepare_command.append("--release-mode")
    run_checked(prepare_command, cwd=REPO_DIR)

    run_checked([
        sys.executable, "-m", "qmini.cli", "audit",
        "--config", str(TOKENIZER_CONFIG_PATH),
    ], cwd=REPO_DIR, capture_output=False)

    parameter_count = active_config.estimated_parameter_count()
    if parameter_count > 1_000_000:
        raise RuntimeError("The selected profile exceeds this notebook's single-device Colab limit.")

    LOCAL_INITIAL_DIR = LOCAL_PACKAGE_DIR / "initial_brain"
    run_checked([
        sys.executable, "-m", "qmini.cli", "build-initial",
        "--config", str(TOKENIZER_CONFIG_PATH),
        "--output-dir", str(LOCAL_INITIAL_DIR),
        "--source-revision", SOURCE_REVISION,
    ], cwd=REPO_DIR)

    # Bind the initial weights to the exact tokenizer and prepared corpus.
    manifest_path = LOCAL_INITIAL_DIR / "model_manifest.json"
    model_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    dataset_manifest_path = DATASET_DIR / "dataset.json"
    dataset_manifest = json.loads(dataset_manifest_path.read_text(encoding="utf-8"))
    model_manifest.update({
        "tokenizer_included": True,
        "tokenizer_relative_path": "../tokenizer/tokenizer.json",
        "tokenizer_sha256": sha256_file(TOKENIZER_PATH),
        "tokenizer_manifest_sha256": sha256_file(Path(str(TOKENIZER_PATH) + ".qmini.json")),
        "prepared_dataset_relative_path": "../dataset",
        "prepared_dataset_manifest_sha256": sha256_file(dataset_manifest_path),
        "train_tokens_sha256": dataset_manifest["train_tokens_sha256"],
        "valid_tokens_sha256": dataset_manifest["valid_tokens_sha256"],
        "train_mask_sha256": dataset_manifest.get("train_mask_sha256"),
        "valid_mask_sha256": dataset_manifest.get("valid_mask_sha256"),
        "corpus_sha256": dataset_manifest.get("corpus_sha256"),
        "source_provenance_sha256": dataset_manifest.get("source_provenance_sha256"),
        "release_mode": bool(dataset_manifest.get("release_mode", False)),
    })
    manifest_path.write_text(
        json.dumps(model_manifest, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )

    # Add an inspectable build report without changing model weights.
    build_report = {
        "format": "qmini-training-ready-build-report-v1",
        "package_key": PACKAGE_KEY,
        "source_revision": SOURCE_REVISION,
        "config": active_config.to_dict(),
        "parameters": parameter_count,
        "architecture": {
            "causal_depthwise_convolution": True,
            "gated_diagonal_state_space": bool(active_config.use_state_space),
            "sparse_top1_moe": active_config.n_experts > 1 and active_config.top_k == 1,
            "shared_refinement_block": True,
            "transformer_self_attention": False,
            "qkv_attention": False,
        },
        "initial_model": {
            "format": "qmini-packed-v2",
            "matrix_weight_bits": 2,
            "sha256": sha256_file(LOCAL_INITIAL_DIR / "model.qmini.npz"),
            "bytes": (LOCAL_INITIAL_DIR / "model.qmini.npz").stat().st_size,
        },
        "tokenizer": {
            "path": "tokenizer/tokenizer.json",
            "vocabulary_size": tokenizer.get_vocab_size(),
            "sha256": sha256_file(TOKENIZER_PATH),
        },
        "dataset": {
            "path": "dataset/",
            "manifest_sha256": sha256_file(dataset_manifest_path),
            "train_tokens": int(dataset_manifest["train_tokens"]),
            "valid_tokens": int(dataset_manifest["valid_tokens"]),
            "loss_mask_mode": dataset_manifest["loss_mask_mode"],
            "release_mode": bool(dataset_manifest.get("release_mode", False)),
        },
        "source_corpus_fingerprint": PACKAGE_KEY,
        "training": {
            "started": False,
            "training_steps": 0,
            "optimizer_steps": 0,
            "tokens_seen": 0,
        },
        "precision": {
            "matrix_weights": "Q2-v2 / 2-bit symbols",
            "row_scales": "FP32",
            "one_dimensional_parameters": "FP16 in packed export",
            "later_QAT_master_parameters": "FP32",
            "later_AdamW_state": "FP32",
            "activations": "signed INT8 fake quantization",
        },
    }
    (LOCAL_PACKAGE_DIR / "BUILD_REPORT.json").write_text(
        json.dumps(build_report, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    # Bundle the complete Q-Mini training source, but keep it dormant.
    runtime_dir = LOCAL_PACKAGE_DIR / "training_runtime"
    runtime_dir.mkdir(parents=True, exist_ok=False)
    source_package = runtime_dir / "qmini"
    shutil.copytree(
        REPO_DIR / "qmini",
        source_package,
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.so", "*.pyd"),
    )
    shutil.copy2(REPO_DIR / "pyproject.toml", runtime_dir / "pyproject.toml")
    training_defaults = {
        "format": "qmini-training-defaults-v1",
        "enabled": False,
        "batch_size": 1,
        "sequence_length": TRAIN_SEQUENCE_LENGTH,
        "gradient_accumulation": 8,
        "learning_rate": 3e-4,
        "weight_decay": 0.1,
        "checkpoint_every_optimizer_steps": 10,
        "evaluation_every_optimizer_steps": 100,
        "total_optimizer_steps": 20000,
        "minimum_free_gpu_gib": 8.0,
    }
    (runtime_dir / "TRAINING_DEFAULTS.json").write_text(
        json.dumps(training_defaults, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    (runtime_dir / "README.md").write_text(
        "# Bundled Q-Mini training runtime\n\n"
        "This runtime is packaged with the initial model but disabled by default. "
        "Activate it only through the separate QMini_Training_Runner.ipynb notebook. "
        "The checkpoint manager writes verified immutable Drive generations and "
        "updates one current-step progress record.\n",
        encoding="utf-8",
    )
    source_files = []
    for source_file in sorted(
        path for path in runtime_dir.rglob("*")
        if path.is_file() and path.name != "TRAINING_RUNTIME.json"
    ):
        source_files.append({
            "path": source_file.relative_to(runtime_dir).as_posix(),
            "sha256": sha256_file(source_file),
            "size_bytes": source_file.stat().st_size,
        })
    runtime_manifest = {
        "format": "qmini-training-runtime-v1",
        "enabled": False,
        "requires_manual_activation": True,
        "entrypoint": "qmini.train.train_model",
        "checkpoint_backend": "qmini.checkpointing",
        "checkpoint_policy": {
            "progress_record": "training_state/training_progress.json",
            "drive_snapshots_retained": 2,
            "publish_after_each_saved_checkpoint": True,
            "verify_sha256_and_torch_archive_crc": True,
            "restore_previous_verified_generation_on_corruption": True,
            "concurrent_session_lease": True,
        },
        "source_revision": SOURCE_REVISION,
        "source_files": source_files,
    }
    (runtime_dir / "TRAINING_RUNTIME.json").write_text(
        json.dumps(runtime_manifest, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    state_dir = LOCAL_PACKAGE_DIR / "training_state"
    state_dir.mkdir(parents=True, exist_ok=False)
    (state_dir / "training_progress.json").write_text(
        json.dumps({
            "format": "qmini-training-progress-v1",
            "status": "not_started",
            "current_step": 0,
            "target_steps": None,
            "progress_percent": 0.0,
            "tokens_seen": 0,
            "updated_at": datetime.now(timezone.utc).isoformat(),
        }, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    model_manifest["training_runtime_included"] = True
    model_manifest["training_runtime_enabled"] = False
    model_manifest["training_runtime_manifest_sha256"] = sha256_file(
        runtime_dir / "TRAINING_RUNTIME.json"
    )
    manifest_path.write_text(
        json.dumps(model_manifest, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    build_report["training_runtime"] = {
        "included": True,
        "enabled": False,
        "source_file_count": len(source_files),
        "manifest_sha256": sha256_file(runtime_dir / "TRAINING_RUNTIME.json"),
        "separate_activation_notebook": "QMini_Training_Runner.ipynb",
        "checkpoint_store": "training_state/",
    }
    (LOCAL_PACKAGE_DIR / "BUILD_REPORT.json").write_text(
        json.dumps(build_report, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    # Validate the local package before copying it to Drive.
    verified = verify_package(LOCAL_PACKAGE_DIR)

    # Publish through a staging directory. Never overwrite an earlier package.
    STAGING_DIR = FINAL_PACKAGE_DIR.parent / f".{PACKAGE_NAME}.staging-{int(time.time())}"
    if STAGING_DIR.exists():
        raise FileExistsError(f"Staging directory already exists: {STAGING_DIR}")
    shutil.copytree(LOCAL_PACKAGE_DIR, STAGING_DIR)
    verify_package(STAGING_DIR)
    STAGING_DIR.rename(FINAL_PACKAGE_DIR)
    PACKAGE_DIR = FINAL_PACKAGE_DIR

# Re-verify the published Drive copy, not just the local source.
verified = verify_package(PACKAGE_DIR)
print("\nQ-MINI TRAINING-READY PACKAGE VERIFIED")
print("Drive package:", PACKAGE_DIR)
print("Parameters:", f"{verified['config'].estimated_parameter_count():,}")
print("Tokenizer vocabulary:", verified["config"].vocab_size)
print("Q2-v2 artifact bytes:", verified["packed_report"].get("archive_bytes"))
print("Training tokens:", f"{verified['train_tokens']:,}")
print("Validation tokens:", f"{verified['valid_tokens']:,}")
print("Neural optimizer steps:", verified["model_manifest"]["optimizer_steps"])
print("Model weight updates:", verified["model_manifest"]["training_steps"])
print("No neural-model training was run in this build stage.")

In [ ]:
# ---------------- BUILD-ONLY FINAL GUARD ----------------
# This cell intentionally cannot launch training.
package_manifest = json.loads(
    (PACKAGE_DIR / "initial_brain" / "model_manifest.json").read_text(encoding="utf-8")
)
runtime_manifest = json.loads(
    (PACKAGE_DIR / "training_runtime" / "TRAINING_RUNTIME.json").read_text(encoding="utf-8")
)
progress = json.loads(
    (PACKAGE_DIR / "training_state" / "training_progress.json").read_text(encoding="utf-8")
)
assert package_manifest["training_steps"] == 0
assert package_manifest["optimizer_steps"] == 0
assert package_manifest["training_runtime_included"] is True
assert package_manifest["training_runtime_enabled"] is False
assert runtime_manifest["enabled"] is False
assert runtime_manifest["requires_manual_activation"] is True
assert progress["status"] == "not_started"
assert progress["current_step"] == 0

print("MODEL BUILD COMPLETE — TRAINING REMAINS DISABLED")
print("Drive package:", PACKAGE_DIR)
print("Initial model SHA-256:", package_manifest["packed_export_sha256"])
print("Bundled training runtime:", PACKAGE_DIR / "training_runtime")
print("Progress record:", PACKAGE_DIR / "training_state" / "training_progress.json")
print("To train later, open the separate QMini_Training_Runner.ipynb notebook.")

## Separate build and training lifecycle

The builder is build-only. It does not create an optimizer, run backward passes, or update model weights. The completed Drive package contains the initial Q2-v2 model, matching tokenizer and prepared streams, plus a checksum-verified copy of the Q-Mini training source in `training_runtime/`. That runtime is marked `enabled: false`.

Training starts only from the separate [QMini Training Runner](https://colab.research.google.com/github/lion36508/Q-mini-light-0.7M/blob/main/colab/QMini_Training_Runner.ipynb). Its checkpoint store updates one visible `training_progress.json` record, keeps two verified recovery generations, and restores the newest valid checkpoint after a runtime/server interruption.

Q2-v2 stores matrix symbols at 2 bits; row scales are FP32, vector parameters FP16, and the current QAT training master weights and AdamW optimizer state are FP32. The Light runner targets the 0.7M-class profile on a single suitable GPU; 5B training stays separate. A future FSDP/ZeRO backend can be replaced independently without changing the model artifact format.